# PairwiseGRU Experiments

This notebook investigates a PairwiseGRU architecture for abductive commonsense reasoning.

Unlike the standard GRU, each hypothesis is evaluated using the same shared scoring network. This design aims to reduce sensitivity to hypothesis order.

The model uses frozen 300-dimensional fastText embeddings.

## Setup and Model Initialization

The ART datasets and pretrained fastText embeddings are loaded.

Initial configuration:
- GRU layers: 2
- Hidden dimension: 128
- Dropout: 0.0
- Learning rate: 0.001
- Batch size: 32
- Seed: 42

We verify the forward pass, CUDA availability, and frozen embeddings.

In [1]:
import sys
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import wandb

from datasets import load_dataset

# Locate repository root
repo_root = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "project1" / "artifacts" / "fasttext_art.pt").exists()
    ),
    None
)

assert repo_root is not None, "Project root not found"

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from project1.src.models import PairwiseGRU
from project1.src.data_loader import create_dataloader
from project1.src.training import train_one_epoch, evaluate

# Reproducibility
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# GPU is required for this tuning notebook
assert torch.cuda.is_available(), "CUDA is not available"

device = torch.device("cuda")

print("=== GPU ===")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

# Load train and validation data
train_data = load_dataset(
    "amarfurt/art-nlp-hs26",
    split="train"
)

val_data = load_dataset(
    "amarfurt/art-nlp-hs26",
    split="validation"
)

# Load verified fastText artifact
artifact_path = (
    repo_root / "project1" / "artifacts" / "fasttext_art.pt"
)

saved = torch.load(
    artifact_path,
    map_location="cpu",
    weights_only=True
)

vocab = saved["vocab"]
embedding_weights = saved["embedding_weights"]

assert embedding_weights.shape == (15038, 300)
assert vocab["[PAD]"] == 0

# Fixed settings for initial tuning
config = {
    "architecture": "PairwiseGRU",
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.0,
    "learning_rate": 0.001,
    "batch_size": 32,
    "max_epochs": 60,
    "patience": 15,
    "seed": seed
}

# Existing preprocessing and DataLoaders
train_loader = create_dataloader(
    train_data,
    vocab,
    batch_size=config["batch_size"],
    shuffle=True
)

val_loader = create_dataloader(
    val_data,
    vocab,
    batch_size=config["batch_size"],
    shuffle=False
)

# Model factory: fresh PairwiseGRU for each experiment
def create_pairwise_model(hidden_dim=128, dropout=0.0):

    embedding_layer = nn.Embedding.from_pretrained(
        embedding_weights.detach().cpu().clone(),
        freeze=True,
        padding_idx=vocab["[PAD]"]
    )

    model = PairwiseGRU(
        embedding_layer=embedding_layer,
        pad_id=vocab["[PAD]"],
        sep_id=vocab["[SEP]"],
        hidden_dim=hidden_dim,
        num_layers=config["num_layers"],
        dropout=dropout
    )

    return model.to(device)


# ============================================================
# PairwiseGRU Initialization Verification
# ============================================================

test_model = create_pairwise_model(
    hidden_dim=config["hidden_dim"],
    dropout=config["dropout"]
)

batch = next(iter(val_loader))

test_model.eval()

with torch.inference_mode():
    logits = test_model(
        batch["input_ids"].to(device),
        lengths=batch["lengths"]
    )

assert logits.shape == (batch["labels"].size(0), 2)
assert logits.device.type == "cuda"
assert torch.isfinite(logits).all().item()
assert not test_model.embedding.weight.requires_grad

trainable_params = sum(
    p.numel()
    for p in test_model.parameters()
    if p.requires_grad
)

print("\n=== PAIRWISE GRU SETUP ===")
print("Architecture:", config["architecture"])
print("Training samples:", len(train_data))
print("Validation samples:", len(val_data))
print("Vocabulary size:", len(vocab))
print("Output shape:", tuple(logits.shape))
print("Device:", logits.device)
print("Trainable parameters:", trainable_params)
print("Frozen embeddings: PASS")
print("Forward pass: PASS")


C:\Users\kimbo\anaconda3\envs\nlp-art-hs26\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== GPU ===
PyTorch: 2.14.1+cu130
CUDA: 13.0
GPU: NVIDIA GeForce RTX 4050 Laptop GPU

=== PAIRWISE GRU SETUP ===
Architecture: PairwiseGRU
Training samples: 16801
Validation samples: 1000
Vocabulary size: 15038
Output shape: (32, 2)
Device: cuda:0
Trainable parameters: 280833
Frozen embeddings: PASS
Forward pass: PASS


## Hypothesis-Swap Invariance Test

We verify that the PairwiseGRU treats both hypotheses symmetrically.

Using 32 validation examples, we exchange H1 and H2 and invert the labels.

The model should produce the same two scores in reversed order.

**Result:** All checks passed.

This confirms that the architecture is invariant to hypothesis ordering, within numerical tolerance. The test uses an untrained model to verify the architectural property.

In [2]:

# Use the same 32 validation examples in both orders
sample_data = val_data.select(range(32))

def swap_hypotheses(example):
    return {
        "hypothesis_1": example["hypothesis_2"],
        "hypothesis_2": example["hypothesis_1"],
        "label": 3 - int(example["label"])
    }

swapped_data = sample_data.map(
    swap_hypotheses,
    load_from_cache_file=False
)

# Create matching DataLoaders
original_loader = create_dataloader(
    sample_data,
    vocab,
    batch_size=32,
    shuffle=False
)

swapped_loader = create_dataloader(
    swapped_data,
    vocab,
    batch_size=32,
    shuffle=False
)

original_batch = next(iter(original_loader))
swapped_batch = next(iter(swapped_loader))

test_model.eval()

with torch.inference_mode():

    original_scores = test_model(
        original_batch["input_ids"].to(device),
        lengths=original_batch["lengths"]
    )

    swapped_scores = test_model(
        swapped_batch["input_ids"].to(device),
        lengths=swapped_batch["lengths"]
    )

# Confirm labels are inverted
assert torch.equal(
    swapped_batch["labels"],
    1 - original_batch["labels"]
)

# Scores must exchange positions when H1 and H2 swap
torch.testing.assert_close(
    original_scores,
    swapped_scores.flip(dims=[1]),
    atol=1e-5,
    rtol=1e-5
)

print("=== PAIRWISE GRU VERIFICATION ===")
print("Original scores shape:", tuple(original_scores.shape))
print("Swapped scores shape:", tuple(swapped_scores.shape))
print("Frozen embeddings: PASS")
print("Hypothesis swapping: PASS")
print("Score-swap invariance: PASS")

print("\nExample 1:")
print("Original scores:", original_scores[0].cpu().tolist())
print("Swapped scores:", swapped_scores[0].cpu().tolist())


Map: 100%|██████████| 32/32 [00:00<00:00, 495.28 examples/s]


=== PAIRWISE GRU VERIFICATION ===
Original scores shape: (32, 2)
Swapped scores shape: (32, 2)
Frozen embeddings: PASS
Hypothesis swapping: PASS
Score-swap invariance: PASS

Example 1:
Original scores: [-0.03029019944369793, -0.03028962016105652]
Swapped scores: [-0.03028962016105652, -0.03029019944369793]


## Multi-Seed PairwiseGRU Training

We train the PairwiseGRU using three random seeds (42, 123, 2026) to evaluate performance stability.

Training configuration:
- Hidden dimension: 128
- GRU layers: 2
- Learning rate: 0.001
- Dropout: 0.0
- Optimizer: Adam
- Frozen fastText embeddings

The best checkpoint is selected using validation accuracy, with validation loss as a tiebreaker. Experiments are tracked in W&B.

In [3]:

import statistics

seeds = [42, 123, 2026]

checkpoint_dir = repo_root / "project1" / "artifacts"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

pairwise_results = []

for seed in seeds:

    print(f"\n{'=' * 60}")
    print(f"PAIRWISE GRU | Seed {seed}")
    print(f"{'=' * 60}")

    # 1. Reset random seeds
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 2. Fresh model
    experiment_model = create_pairwise_model(
        hidden_dim=config["hidden_dim"],
        dropout=config["dropout"]
    )

    optimizer = torch.optim.Adam(
        [p for p in experiment_model.parameters()
         if p.requires_grad],
        lr=config["learning_rate"],
        weight_decay=0.0
    )

    criterion = nn.CrossEntropyLoss()

    # Fresh DataLoader
    experiment_train_loader = create_dataloader(
        train_data,
        vocab,
        batch_size=config["batch_size"],
        shuffle=True
    )

    # 3. W&B setup
    run_name = (
        f"pairwise_gru_h128_l2_lr0p001_"
        f"drop0p0_seed{seed}_cuda"
    )

    run = wandb.init(
        entity="kimbortis-hochschule-luzern",
        project="nlp-art-hs26-project1",
        group="pairwise-gru",
        job_type="architecture-comparison",
        name=run_name,
        config={
            **config,
            "seed": seed,
            "optimizer": "Adam",
            "weight_decay": 0.0,
            "embedding": "fasttext-wiki-news-subwords-300",
            "embedding_frozen": True,
            "preprocessing": "v2",
            "candidate_format": "O1 SEP H SEP O2",
            "shared_candidate_scorer": True,
            "device": str(device),
            "gpu_name": torch.cuda.get_device_name(0)
        }
    )

    run.define_metric("epoch")
    run.define_metric("train/*", step_metric="epoch")
    run.define_metric("val/*", step_metric="epoch")

    checkpoint_path = (
        checkpoint_dir / f"{run_name}_{run.id}_best.pt"
    )

    best_accuracy = -1.0
    best_loss = float("inf")
    lowest_loss = float("inf")
    best_epoch = 0
    no_improvement = 0
    last_epoch = 0

    # Match RNG convention of the previous GRU experiments
    random.seed(seed + 1)
    np.random.seed(seed + 1)
    torch.manual_seed(seed + 1)
    torch.cuda.manual_seed_all(seed + 1)

    try:
        # 4. Training loop
        for epoch in range(1, config["max_epochs"] + 1):
            last_epoch = epoch

            train_metrics = train_one_epoch(
                experiment_model,
                experiment_train_loader,
                optimizer,
                criterion,
                device
            )

            val_metrics = evaluate(
                experiment_model,
                val_loader,
                criterion,
                device
            )

            run.log({
                "epoch": epoch,
                "train/loss": train_metrics["loss"],
                "train/accuracy": train_metrics["accuracy"],
                "val/loss": val_metrics["loss"],
                "val/accuracy": val_metrics["accuracy"]
            })

            lowest_loss = min(
                lowest_loss,
                val_metrics["loss"]
            )

            # Best accuracy; validation loss breaks ties
            improved = (
                val_metrics["accuracy"] > best_accuracy
                or (
                    val_metrics["accuracy"] == best_accuracy
                    and val_metrics["loss"] < best_loss
                )
            )

            if improved:
                best_accuracy = val_metrics["accuracy"]
                best_loss = val_metrics["loss"]
                best_epoch = epoch
                no_improvement = 0

                torch.save(
                    experiment_model.state_dict(),
                    checkpoint_path
                )
            else:
                no_improvement += 1

            if epoch == 1 or epoch % 5 == 0 or improved:
                print(
                    f"Epoch {epoch:02d} | "
                    f"Train: {train_metrics['accuracy']:.2%} | "
                    f"Val: {val_metrics['accuracy']:.2%} | "
                    f"Val loss: {val_metrics['loss']:.4f}"
                )

            if no_improvement >= config["patience"]:
                print(f"Early stopping at epoch {epoch}")
                break

        # 5. Restore best checkpoint
        experiment_model.load_state_dict(
            torch.load(
                checkpoint_path,
                map_location=device,
                weights_only=True
            )
        )

        restored = evaluate(
            experiment_model,
            val_loader,
            criterion,
            device
        )

        assert abs(
            restored["accuracy"] - best_accuracy
        ) < 1e-6

        assert abs(
            restored["loss"] - best_loss
        ) < 1e-5

        # 6. W&B summary
        run.summary["best_val_accuracy"] = best_accuracy
        run.summary["best_epoch"] = best_epoch
        run.summary["val_loss_at_best_accuracy"] = best_loss
        run.summary["lowest_val_loss"] = lowest_loss
        run.summary["epochs_trained"] = last_epoch
        run.summary["checkpoint"] = str(checkpoint_path)

        pairwise_results.append({
            "seed": seed,
            "best_accuracy": best_accuracy,
            "best_epoch": best_epoch,
            "best_loss": best_loss,
            "lowest_val_loss": lowest_loss,
            "epochs_trained": last_epoch,
            "checkpoint": str(checkpoint_path)
        })

        print("\n=== PAIRWISE GRU RESULT ===")
        print("Seed:", seed)
        print(f"Best validation accuracy: {best_accuracy:.2%}")
        print(f"Best epoch: {best_epoch}")
        print(f"Loss at best: {best_loss:.4f}")
        print(f"Lowest validation loss: {lowest_loss:.4f}")
        print("Checkpoint verification: PASS")

    finally:
        run.finish()
        del experiment_model, optimizer
        torch.cuda.empty_cache()


# ============================================================
# Architecture Comparison
# ============================================================

baseline_gru_scores = {
    42: 0.685,
    123: 0.677,
    2026: 0.679
}

print("\n=== PAIRWISE GRU MULTI-SEED COMPARISON ===")

for result in pairwise_results:
    seed = result["seed"]
    baseline = baseline_gru_scores[seed]
    pairwise = result["best_accuracy"]

    print(
        f"Seed {seed}: "
        f"Standard GRU={baseline:.2%} | "
        f"Pairwise GRU={pairwise:.2%} | "
        f"Difference={100 * (pairwise - baseline):+.2f} pp"
    )

pairwise_scores = [
    r["best_accuracy"] for r in pairwise_results
]

baseline_mean = statistics.mean(
    baseline_gru_scores.values()
)
pairwise_mean = statistics.mean(pairwise_scores)

print("\n=== AGGREGATED ARCHITECTURE RESULTS ===")
print(f"Standard GRU mean: {baseline_mean:.2%}")
print(f"Pairwise GRU mean: {pairwise_mean:.2%}")
print(
    f"Mean difference: "
    f"{100 * (pairwise_mean - baseline_mean):+.2f} pp"
)
print(
    f"Pairwise standard deviation: "
    f"{statistics.stdev(pairwise_scores):.2%}"
)
print(
    f"Best Pairwise GRU run: "
    f"{max(pairwise_scores):.2%}"
)



PAIRWISE GRU | Seed 42


wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\kimbo\_netrc.
wandb: Currently logged in as: kimbortis (kimbortis-hochschule-luzern) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Epoch 01 | Train: 62.71% | Val: 63.80% | Val loss: 0.6263
Epoch 03 | Train: 68.41% | Val: 67.00% | Val loss: 0.5994
Epoch 04 | Train: 69.42% | Val: 68.10% | Val loss: 0.5968
Epoch 05 | Train: 70.82% | Val: 67.50% | Val loss: 0.5961
Epoch 06 | Train: 72.12% | Val: 68.80% | Val loss: 0.5813
Epoch 07 | Train: 73.68% | Val: 69.00% | Val loss: 0.5828
Epoch 08 | Train: 75.54% | Val: 69.90% | Val loss: 0.5834
Epoch 10 | Train: 79.57% | Val: 71.10% | Val loss: 0.5941
Epoch 15 | Train: 92.08% | Val: 69.30% | Val loss: 1.1170
Epoch 20 | Train: 97.32% | Val: 68.40% | Val loss: 2.1085
Epoch 25 | Train: 97.41% | Val: 70.60% | Val loss: 2.0450
Early stopping at epoch 25

=== PAIRWISE GRU RESULT ===
Seed: 42
Best validation accuracy: 71.10%
Best epoch: 10
Loss at best: 0.5941
Lowest validation loss: 0.5813
Checkpoint verification: PASS


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
train/accuracy,▁▂▂▂▃▃▃▄▄▄▅▅▆▆▇▇▇████████
train/loss,██▇▇▇▇▆▆▆▅▅▄▄▃▃▂▂▂▁▁▁▁▁▁▁
val/accuracy,▁▁▄▅▅▆▆▇▆█▆██▇▆▇▇▆▇▅▇▆▇▇█
val/loss,▁▁▁▁▁▁▁▁▁▁▁▂▂▂▃▄▄▅▅▆█▇█▇▆
best_epoch,10
best_val_accuracy,0.711
checkpoint,C:\Users\kimbo\Docum...
epoch,25
epochs_trained,25
lowest_val_loss,0.58126



PAIRWISE GRU | Seed 123


Epoch 01 | Train: 63.40% | Val: 63.50% | Val loss: 0.6378
Epoch 02 | Train: 66.67% | Val: 66.50% | Val loss: 0.6139
Epoch 03 | Train: 68.31% | Val: 67.20% | Val loss: 0.6017
Epoch 04 | Train: 69.72% | Val: 67.40% | Val loss: 0.5949
Epoch 05 | Train: 70.94% | Val: 67.40% | Val loss: 0.5956
Epoch 07 | Train: 73.38% | Val: 69.40% | Val loss: 0.5931
Epoch 08 | Train: 75.10% | Val: 70.10% | Val loss: 0.5850
Epoch 10 | Train: 79.33% | Val: 70.60% | Val loss: 0.6204
Epoch 15 | Train: 91.30% | Val: 68.80% | Val loss: 1.1229
Epoch 20 | Train: 97.54% | Val: 71.40% | Val loss: 2.0846
Epoch 25 | Train: 98.38% | Val: 70.10% | Val loss: 2.3715
Epoch 30 | Train: 98.92% | Val: 67.80% | Val loss: 2.3834
Epoch 35 | Train: 99.17% | Val: 69.10% | Val loss: 3.2172
Early stopping at epoch 35

=== PAIRWISE GRU RESULT ===
Seed: 123
Best validation accuracy: 71.40%
Best epoch: 20
Loss at best: 2.0846
Lowest validation loss: 0.5850
Checkpoint verification: PASS


epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
train/accuracy,▁▂▂▂▂▃▃▃▄▄▅▅▆▆▆▇▇▇▇████████████████
train/loss,██▇▇▇▇▇▆▆▆▅▅▄▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/accuracy,▁▄▄▄▄▄▆▇▆▇▇▆▇▆▆▅▅▆▆█▇▇▇▆▇▅▆▆▇▅▇▇▆▇▆
val/loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▃▄▄▅▅▄▅▆▆▅▆▆▆▆▅▆▆█▆▇
best_epoch,20
best_val_accuracy,0.714
checkpoint,C:\Users\kimbo\Docum...
epoch,35
epochs_trained,35
lowest_val_loss,0.58501



PAIRWISE GRU | Seed 2026


Epoch 01 | Train: 62.03% | Val: 63.60% | Val loss: 0.6234
Epoch 02 | Train: 66.57% | Val: 64.40% | Val loss: 0.6150
Epoch 03 | Train: 68.10% | Val: 66.30% | Val loss: 0.6065
Epoch 04 | Train: 69.59% | Val: 68.00% | Val loss: 0.5959
Epoch 05 | Train: 70.51% | Val: 69.00% | Val loss: 0.5965
Epoch 06 | Train: 71.69% | Val: 70.80% | Val loss: 0.5805
Epoch 10 | Train: 78.22% | Val: 71.00% | Val loss: 0.5776
Epoch 11 | Train: 80.14% | Val: 71.10% | Val loss: 0.6391
Epoch 12 | Train: 83.07% | Val: 71.30% | Val loss: 0.6646
Epoch 15 | Train: 89.81% | Val: 70.60% | Val loss: 0.9815
Epoch 20 | Train: 96.83% | Val: 70.70% | Val loss: 2.0593
Epoch 25 | Train: 98.45% | Val: 70.50% | Val loss: 2.5086
Early stopping at epoch 27

=== PAIRWISE GRU RESULT ===
Seed: 2026
Best validation accuracy: 71.30%
Best epoch: 12
Loss at best: 0.6646
Lowest validation loss: 0.5776
Checkpoint verification: PASS


epoch,▁▁▂▂▂▂▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇██
train/accuracy,▁▂▂▂▃▃▃▃▄▄▄▅▅▆▆▇▇▇▇████████
train/loss,█▇▇▇▇▇▇▆▆▆▅▅▄▄▃▃▂▂▂▂▁▁▁▁▁▁▁
val/accuracy,▁▂▃▅▆█▆▇▇█████▇▇███▇▆█▇▆▇▇▇
val/loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▃▃▄▄▅▅▅▆▆▇▆█
best_epoch,12
best_val_accuracy,0.713
checkpoint,C:\Users\kimbo\Docum...
epoch,27
epochs_trained,27
lowest_val_loss,0.57765



=== PAIRWISE GRU MULTI-SEED COMPARISON ===
Seed 42: Standard GRU=68.50% | Pairwise GRU=71.10% | Difference=+2.60 pp
Seed 123: Standard GRU=67.70% | Pairwise GRU=71.40% | Difference=+3.70 pp
Seed 2026: Standard GRU=67.90% | Pairwise GRU=71.30% | Difference=+3.40 pp

=== AGGREGATED ARCHITECTURE RESULTS ===
Standard GRU mean: 68.03%
Pairwise GRU mean: 71.27%
Mean difference: +3.23 pp
Pairwise standard deviation: 0.15%
Best Pairwise GRU run: 71.40%


### Multi-Seed Results

| Seed | Standard GRU | PairwiseGRU |
|---|---:|---:|
| 42 | 68.50% | 71.10% |
| 123 | 67.70% | 71.40% |
| 2026 | 67.90% | 71.30% |
| **Mean** | **68.03%** | **71.27%** |

The PairwiseGRU outperformed the standard GRU on validation data across all three seeds.

- Mean improvement: +3.23 percentage points
- PairwiseGRU standard deviation: 0.15 percentage points

The results suggest improved validation performance and training stability. Generalization to the test set is evaluated separately.

## Validation Diagnostics

We evaluate the best PairwiseGRU checkpoint (seed 42) using a confusion matrix and hypothesis-swapping analysis.

**Results:**
- Validation accuracy: 71.10%
- H1 recall: 72.44%
- H2 recall: 69.72%
- Prediction-flip consistency: 99.80%

Two examples produced identical scores:
- Example 441: Both distinguishing words map to `[UNK]`.
- Example 988: Both hypotheses are identical.

Excluding these ties, prediction-flip consistency is 100%.

This confirms that the PairwiseGRU is symmetric with respect to hypothesis order, within numerical tolerance.

In [4]:

checkpoint_path = (
    repo_root / "project1" / "artifacts" /
    "pairwise_gru_h128_l2_lr0p001_drop0p0_seed42_cuda_z6d879wl_best.pt"
)

assert checkpoint_path.exists(), checkpoint_path

diagnostic_model = create_pairwise_model(
    hidden_dim=128,
    dropout=0.0
)

diagnostic_model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )
)

diagnostic_model.eval()

# Swap H1 and H2 in all validation examples
def swap_example(example):
    return {
        "hypothesis_1": example["hypothesis_2"],
        "hypothesis_2": example["hypothesis_1"],
        "label": 3 - int(example["label"])
    }

swapped_val_data = val_data.map(
    swap_example,
    load_from_cache_file=False
)

swapped_loader = create_dataloader(
    swapped_val_data,
    vocab,
    batch_size=config["batch_size"],
    shuffle=False
)

def collect_outputs(model, loader):
    all_logits = []
    all_labels = []

    model.eval()

    with torch.inference_mode():
        for batch in loader:
            logits = model(
                batch["input_ids"].to(device),
                lengths=batch["lengths"]
            )

            all_logits.append(logits.cpu())
            all_labels.append(batch["labels"].cpu())

    return torch.cat(all_logits), torch.cat(all_labels)


original_logits, labels = collect_outputs(
    diagnostic_model, val_loader
)

swapped_logits, swapped_labels = collect_outputs(
    diagnostic_model, swapped_loader
)

assert torch.equal(swapped_labels, 1 - labels)

original_pred = original_logits.argmax(dim=1)
swapped_pred = swapped_logits.argmax(dim=1)

# Confusion matrix
confusion = torch.bincount(
    labels * 2 + original_pred,
    minlength=4
).reshape(2, 2)

recalls = (
    confusion.diag().float()
    / confusion.sum(dim=1).clamp(min=1)
)

accuracy = (
    original_pred == labels
).float().mean().item()

swapped_accuracy = (
    swapped_pred == swapped_labels
).float().mean().item()

flip_consistency = (
    original_pred == (1 - swapped_pred)
).float().mean().item()

balanced_accuracy = recalls.mean().item()

max_score_difference = (
    original_logits - swapped_logits.flip(dims=[1])
).abs().max().item()

print("=== PAIRWISE GRU VALIDATION DIAGNOSTICS ===")
print(f"Accuracy: {accuracy:.2%}")
print(f"Balanced accuracy: {balanced_accuracy:.2%}")
print(f"Swapped accuracy: {swapped_accuracy:.2%}")
print(f"Prediction-flip consistency: {flip_consistency:.2%}")
print(f"Maximum swapped-score difference: {max_score_difference:.8f}")

print("\nConfusion matrix (true rows, predicted columns):")
print(confusion)

print("\nPer-class recall:")
print(f"H1: {recalls[0].item():.2%}")
print(f"H2: {recalls[1].item():.2%}")

print("\nPrediction distribution:")
print(f"Predicted H1: {(original_pred == 0).float().mean().item():.2%}")
print(f"Predicted H2: {(original_pred == 1).float().mean().item():.2%}")

assert abs(accuracy - 0.711) < 1e-6

print("\nCheckpoint verification: PASS")


Map: 100%|██████████| 1000/1000 [00:00<00:00, 4493.52 examples/s]


=== PAIRWISE GRU VALIDATION DIAGNOSTICS ===
Accuracy: 71.10%
Balanced accuracy: 71.08%
Swapped accuracy: 71.10%
Prediction-flip consistency: 99.80%
Maximum swapped-score difference: 0.00000000

Confusion matrix (true rows, predicted columns):
tensor([[368, 140],
        [149, 343]])

Per-class recall:
H1: 72.44%
H2: 69.72%

Prediction distribution:
Predicted H1: 51.70%
Predicted H2: 48.30%

Checkpoint verification: PASS


In [5]:

# Investigate the 0.2% of predictions that did not flip

no_flip = original_pred == swapped_pred

indices = torch.where(no_flip)[0]

print("=== SWAP CONSISTENCY INVESTIGATION ===")
print("Non-flipping predictions:", len(indices))

for idx in indices.tolist():
    print(f"\nValidation example {idx}")
    print("H1:", val_data[idx]["hypothesis_1"])
    print("H2:", val_data[idx]["hypothesis_2"])
    print("Original logits:", original_logits[idx].tolist())
    print("Swapped logits:", swapped_logits[idx].tolist())

    margin = abs(
        original_logits[idx, 0]
        - original_logits[idx, 1]
    ).item()

    print(f"Score difference: {margin:.12f}")

print(
    "\nExact original logit ties:",
    (original_logits[:, 0] == original_logits[:, 1])
    .sum().item()
)

print(
    "Near ties (< 1e-6):",
    ((original_logits[:, 0] - original_logits[:, 1]).abs() < 1e-6)
    .sum().item()
)

torch.testing.assert_close(
    original_logits,
    swapped_logits.flip(dims=[1]),
    atol=1e-6,
    rtol=1e-6
)

print("Score-swap invariance: PASS")


=== SWAP CONSISTENCY INVESTIGATION ===
Non-flipping predictions: 2

Validation example 441
H1: Then she saw a documentary about the unhealthiness of soda.
H2: Then she saw a documentary about the healthiness of soda.
Original logits: [-17.254793167114258, -17.254793167114258]
Swapped logits: [-17.254793167114258, -17.254793167114258]
Score difference: 0.000000000000

Validation example 988
H1: My boyfriend made a very moving proposal.
H2: My boyfriend made a very moving proposal.
Original logits: [-18.66149139404297, -18.66149139404297]
Swapped logits: [-18.66149139404297, -18.66149139404297]
Score difference: 0.000000000000

Exact original logit ties: 2
Near ties (< 1e-6): 2
Score-swap invariance: PASS


In [6]:
# Investigate whether tokenization causes identical inputs

from project1.src.preprocessing import tokenize

unk_id = vocab["[UNK]"]

print("=== TIE TOKENIZATION INVESTIGATION ===")

for idx in [441, 988]:
    example = val_data[idx]

    h1_tokens = tokenize(example["hypothesis_1"])
    h2_tokens = tokenize(example["hypothesis_2"])

    h1_ids = [vocab.get(token, unk_id) for token in h1_tokens]
    h2_ids = [vocab.get(token, unk_id) for token in h2_tokens]

    print(f"\nValidation example {idx}")
    print("H1 tokens:", h1_tokens)
    print("H2 tokens:", h2_tokens)

    print("H1 unknown words:",
          [t for t in h1_tokens if t not in vocab])

    print("H2 unknown words:",
          [t for t in h2_tokens if t not in vocab])

    print("Identical token IDs:", h1_ids == h2_ids)


=== TIE TOKENIZATION INVESTIGATION ===

Validation example 441
H1 tokens: ['then', 'she', 'saw', 'a', 'documentary', 'about', 'the', 'unhealthiness', 'of', 'soda', '.']
H2 tokens: ['then', 'she', 'saw', 'a', 'documentary', 'about', 'the', 'healthiness', 'of', 'soda', '.']
H1 unknown words: ['unhealthiness']
H2 unknown words: ['healthiness']
Identical token IDs: True

Validation example 988
H1 tokens: ['my', 'boyfriend', 'made', 'a', 'very', 'moving', 'proposal', '.']
H2 tokens: ['my', 'boyfriend', 'made', 'a', 'very', 'moving', 'proposal', '.']
H1 unknown words: []
H2 unknown words: []
Identical token IDs: True


## Final PairwiseGRU Analysis

The PairwiseGRU achieved a mean validation accuracy of
**71.27% across three seeds**, improving on the standard
GRU (68.03%) by **3.23 percentage points**.

The architecture also demonstrated exact score-swap
invariance within the tested numerical tolerance.

Prediction-flip consistency was 99.80% because two
validation examples produced identical candidate scores:

- Example 441: The distinguishing words "unhealthiness"
  and "healthiness" were both mapped to `[UNK]`.
- Example 988: Both hypotheses contained identical text.

Excluding these two exact ties, prediction-flip
consistency was 100% (998/998).

### Limitations

- The model still overfits during later training epochs.
- The fixed vocabulary loses distinctions between
  out-of-vocabulary words.
- The validation improvements do not guarantee better test performance.
  Test results are discussed in Notebook 06.

The PairwiseGRU achieved the highest validation accuracy (71.27% mean) and demonstrated strong hypothesis-swap consistency. 
However, its generalization performance must be evaluated separately.


## Checkpoint Inventory and Robustness Analysis

We first list the saved MLP, GRU, and PairwiseGRU checkpoints.

Next, we investigate hypothesis similarity using Jaccard similarity between H1 and H2.

The goal is to evaluate whether models generalize to examples where the two hypotheses have less lexical overlap.

In [7]:
# ============================================================
# Final Evaluation — Checkpoint Inventory
# ============================================================

artifact_dir = repo_root / "project1" / "artifacts"

print("=== AVAILABLE MODEL CHECKPOINTS ===")

for path in sorted(artifact_dir.glob("*.pt")):
    name = path.name.lower()

    if (
        "mlp" in name
        or "gru_multiseed_h128" in name
        or "pairwise_gru" in name
    ):
        print(path.name)

print("\nCheckpoint inventory: COMPLETE")


=== AVAILABLE MODEL CHECKPOINTS ===
gru_multiseed_h128_lr0p001_drop0p0_seed123_cuda_di4x3b3x_best.pt
gru_multiseed_h128_lr0p001_drop0p0_seed2026_cuda_l441q4tm_best.pt
gru_multiseed_h128_lr0p001_drop0p0_seed42_cuda_hvxtunb8_best.pt
mlp_best.pt
mlp_h128_lr0.0003_bs32_drop0p2_adam_wd0p0_seed42_w18vyupc_best.pt
mlp_h128_lr0.0003_bs32_drop0p2_adamw_wd0p0001_seed42_l03fhnc1_best.pt
mlp_h128_lr0.0003_bs32_drop0p2_adamw_wd0p0_seed42_bbvyew7m_best.pt
mlp_h128_lr0.0003_bs32_seed42_drop0p0_preprocV2_r1dgmfuo_best.pt
mlp_h128_lr0.0003_bs32_seed42_drop0p2_preprocV2_yf3ahvuk_best.pt
mlp_h128_lr0.0003_bs32_seed42_preprocV2_ES60_k4h7l9iw_best.pt
mlp_h128_lr0.001_bs32_preprocV2_l513a03a_best.pt
mlp_h128_lr0.001_bs32_seed42_preprocV2_200ep_noES_gyjy8tcu_best.pt
mlp_h128_lr0.001_bs32_seed42_preprocV2_4428gh0d_best.pt
mlp_h128_lr0.001_bs32_seed42_preprocV2_ES60_j8gcvl1v_best.pt
mlp_h128_lr0.003_bs32_seed42_preprocV2_ES60_i9j3c1yl_best.pt
mlp_multiseed_s123_drop0p0_lr0p0003_95l2564a_best.pt
mlp_multiseed_s

In [8]:

# ============================================================
# Robustness Experiment — Training Distribution
# ============================================================

import numpy as np
from project1.src.preprocessing import tokenize

def hypothesis_similarity(example):
    h1 = set(tokenize(example["hypothesis_1"]))
    h2 = set(tokenize(example["hypothesis_2"]))

    union = h1 | h2

    return len(h1 & h2) / len(union) if union else 1.0


# IMPORTANT: Analyze TRAINING data only.
train_similarities = np.array([
    hypothesis_similarity(example)
    for example in train_data
])

print("=== TRAINING HYPOTHESIS SIMILARITY ===")
print("Samples:", len(train_similarities))

for percentile in [0, 5, 10, 20, 25, 50, 75, 90, 100]:
    value = np.percentile(
        train_similarities, percentile
    )
    print(f"{percentile:3d}th percentile: {value:.3f}")


# Candidate stress-validation subset:
# The lowest-similarity 10% of TRAINING examples.
n_hard = int(len(train_data) * 0.10)

hard_indices = np.argsort(
    train_similarities,
    kind="stable"
)[:n_hard]

hard_similarities = train_similarities[hard_indices]

hard_labels = [
    train_data[int(idx)]["label"]
    for idx in hard_indices
]

print("\n=== CANDIDATE ROBUSTNESS SPLIT ===")
print("Examples:", len(hard_indices))
print(f"Mean similarity: {hard_similarities.mean():.3f}")
print(f"Maximum similarity: {hard_similarities.max():.3f}")

print(
    "H1 label proportion:",
    f"{100 * hard_labels.count(1) / len(hard_labels):.2f}%"
)

print("\nAnalysis: PASS")
print("No training or test evaluation performed.")


=== TRAINING HYPOTHESIS SIMILARITY ===
Samples: 16801
  0th percentile: 0.040
  5th percentile: 0.167
 10th percentile: 0.250
 20th percentile: 0.417
 25th percentile: 0.462
 50th percentile: 0.667
 75th percentile: 0.778
 90th percentile: 0.833
100th percentile: 1.000

=== CANDIDATE ROBUSTNESS SPLIT ===
Examples: 1680
Mean similarity: 0.161
Maximum similarity: 0.250
H1 label proportion: 50.06%

Analysis: PASS
No training or test evaluation performed.


### Robustness Setup

| Dataset | Examples | Mean similarity |
|---|---:|---:|
| Development training | 15,121 | 0.623 |
| Normal holdout | 840 | 0.657 |
| Hard holdout | 840 | 0.160 |

Both holdouts are excluded from training.

A separate vocabulary containing 14,440 tokens is constructed using development-training data only.

This setup allows controlled robustness experiments without using the official test set.

In [9]:
# ============================================================
# Experiment 6 — Robustness Development Split
# ============================================================

import numpy as np

rng = np.random.default_rng(42)

# Previously identified lowest-similarity 10%
hard_pool = np.array(hard_indices, dtype=int).copy()
rng.shuffle(hard_pool)

# Reserve half for evaluation; retain half for training
hard_dev_indices = hard_pool[:840]
hard_train_indices = hard_pool[840:]

# Remaining examples have higher hypothesis similarity
other_indices = np.setdiff1d(
    np.arange(len(train_data)),
    hard_pool
)

# Create an independent normal validation subset
normal_dev_indices = rng.choice(
    other_indices,
    size=840,
    replace=False
)

# Exclude both development subsets from training
excluded_indices = set(
    hard_dev_indices.tolist()
    + normal_dev_indices.tolist()
)

development_train_indices = [
    i for i in range(len(train_data))
    if i not in excluded_indices
]

# Hugging Face datasets
robust_train_data = train_data.select(
    development_train_indices
)

hard_dev_data = train_data.select(
    hard_dev_indices.tolist()
)

normal_dev_data = train_data.select(
    normal_dev_indices.tolist()
)

# ============================================================
# Verification
# ============================================================

assert len(robust_train_data) == 15121
assert len(hard_dev_data) == 840
assert len(normal_dev_data) == 840

assert len(excluded_indices) == 1680

# Ensure all reserved low-similarity training examples remain
assert set(hard_train_indices).issubset(
    set(development_train_indices)
)

def describe_split(name, dataset):
    similarities = np.array([
        hypothesis_similarity(row)
        for row in dataset
    ])

    labels = np.array(dataset["label"])

    print(f"\n{name}")
    print("Samples:", len(dataset))
    print(f"Mean similarity: {similarities.mean():.3f}")
    print(f"Median similarity: {np.median(similarities):.3f}")
    print(f"H1 labels: {(labels == 1).mean():.2%}")


print("=== ROBUSTNESS DEVELOPMENT SPLIT ===")

describe_split("Development training", robust_train_data)
describe_split("Hard development holdout", hard_dev_data)
describe_split("Normal development holdout", normal_dev_data)

print("\nRetained hard training examples:",
      len(hard_train_indices))

print("No overlapping split indices: PASS")
print("Original validation/test not used: PASS")
print("Robustness split: PASS")


=== ROBUSTNESS DEVELOPMENT SPLIT ===

Development training
Samples: 15121
Mean similarity: 0.623
Median similarity: 0.667
H1 labels: 50.33%

Hard development holdout
Samples: 840
Mean similarity: 0.160
Median similarity: 0.167
H1 labels: 49.17%

Normal development holdout
Samples: 840
Mean similarity: 0.657
Median similarity: 0.667
H1 labels: 52.14%

Retained hard training examples: 840
No overlapping split indices: PASS
Original validation/test not used: PASS
Robustness split: PASS


In [10]:

# ============================================================
# Experiment 6 — Isolated Development Vocabulary
# ============================================================

from project1.src.preprocessing import FIELDS, tokenize

# ------------------------------------------------------------
# 1. Collect vocabulary from development training ONLY
# ------------------------------------------------------------

development_tokens = set()

for example in robust_train_data:
    for field in FIELDS:
        development_tokens.update(
            tokenize(example[field])
        )

special_tokens = ["[PAD]", "[UNK]", "[SEP]"]

# Preserve original vocabulary order
ordered_original_tokens = sorted(
    vocab.keys(),
    key=lambda token: vocab[token]
)

allowed_tokens = [
    token
    for token in ordered_original_tokens
    if (
        token in special_tokens
        or token in development_tokens
    )
]

# New contiguous token IDs
dev_vocab = {
    token: idx
    for idx, token in enumerate(allowed_tokens)
}

# Reuse ONLY pretrained vectors for allowed tokens
original_ids = [
    vocab[token]
    for token in allowed_tokens
]

dev_embedding_weights = (
    embedding_weights[original_ids]
    .detach()
    .cpu()
    .clone()
)

# ------------------------------------------------------------
# 2. Verify vocabulary isolation
# ------------------------------------------------------------

assert dev_vocab["[PAD]"] == 0
assert dev_vocab["[UNK]"] == 1
assert dev_vocab["[SEP]"] == 2

assert dev_embedding_weights.shape == (
    len(dev_vocab), 300
)

assert all(
    token in development_tokens
    for token in dev_vocab
    if token not in special_tokens
)

print("=== DEVELOPMENT VOCABULARY ===")
print("Original vocabulary:", len(vocab))
print("Development vocabulary:", len(dev_vocab))
print("Embedding shape:", tuple(dev_embedding_weights.shape))
print("Vocabulary isolation: PASS")


# ------------------------------------------------------------
# 3. Create isolated DataLoaders
# ------------------------------------------------------------

robust_train_loader = create_dataloader(
    robust_train_data,
    dev_vocab,
    batch_size=32,
    shuffle=True
)

normal_dev_loader = create_dataloader(
    normal_dev_data,
    dev_vocab,
    batch_size=32,
    shuffle=False
)

hard_dev_loader = create_dataloader(
    hard_dev_data,
    dev_vocab,
    batch_size=32,
    shuffle=False
)

print("\n=== DEVELOPMENT DATALOADERS ===")
print("Training:", len(robust_train_data))
print("Normal holdout:", len(normal_dev_data))
print("Hard holdout:", len(hard_dev_data))
print("DataLoaders: PASS")


# ------------------------------------------------------------
# 4. New isolated PairwiseGRU factory
# ------------------------------------------------------------

def create_robust_pairwise_model(
    hidden_dim=128,
    dropout=0.0
):

    embedding_layer = nn.Embedding.from_pretrained(
        dev_embedding_weights.clone(),
        freeze=True,
        padding_idx=dev_vocab["[PAD]"]
    )

    return PairwiseGRU(
        embedding_layer=embedding_layer,
        pad_id=dev_vocab["[PAD]"],
        sep_id=dev_vocab["[SEP]"],
        hidden_dim=hidden_dim,
        num_layers=2,
        dropout=dropout
    ).to(device)


# ------------------------------------------------------------
# 5. Forward-pass verification
# ------------------------------------------------------------

test_model = create_robust_pairwise_model(
    hidden_dim=128,
    dropout=0.0
)

batch = next(iter(normal_dev_loader))

test_model.eval()

with torch.inference_mode():
    logits = test_model(
        batch["input_ids"].to(device),
        lengths=batch["lengths"]
    )

assert logits.shape == (
    batch["labels"].size(0), 2
)

assert torch.isfinite(logits).all().item()
assert not test_model.embedding.weight.requires_grad

print("\n=== ROBUST PAIRWISE GRU ===")
print("Output shape:", tuple(logits.shape))
print("Device:", logits.device)
print("Frozen embeddings: PASS")
print("Forward pass: PASS")
print("Ready for controlled training: PASS")

del test_model, logits


=== DEVELOPMENT VOCABULARY ===
Original vocabulary: 15038
Development vocabulary: 14440
Embedding shape: (14440, 300)
Vocabulary isolation: PASS

=== DEVELOPMENT DATALOADERS ===
Training: 15121
Normal holdout: 840
Hard holdout: 840
DataLoaders: PASS

=== ROBUST PAIRWISE GRU ===
Output shape: (32, 2)
Device: cuda:0
Frozen embeddings: PASS
Forward pass: PASS
Ready for controlled training: PASS


## PairwiseGRU Regularization Experiments

We investigate whether regularization improves generalization to low-similarity examples.

Four configurations are compared using the same development split and training settings.

Checkpoints are selected using the normal development holdout. The hard holdout is evaluated only after training to assess robustness.

In [11]:

import random
import numpy as np
import torch
import torch.nn as nn
import wandb

experiments = [
    {
        "name": "baseline",
        "dropout": 0.0,
        "weight_decay": 0.0
    },
    {
        "name": "regularized",
        "dropout": 0.3,
        "weight_decay": 0.0001
    }
]

seed = 42
learning_rate = 0.001
max_epochs = 60
patience = 15

checkpoint_dir = (
    repo_root / "project1" / "artifacts"
)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

robustness_results = []

criterion = nn.CrossEntropyLoss()

# ============================================================
# Train each configuration
# ============================================================

for experiment in experiments:

    name = experiment["name"]

    print(f"\n{'=' * 60}")
    print(f"ROBUSTNESS EXPERIMENT: {name.upper()}")
    print(f"{'=' * 60}")

    # --------------------------------------------------------
    # 1. Reproducibility
    # --------------------------------------------------------

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Fresh model
    model = create_robust_pairwise_model(
        hidden_dim=128,
        dropout=experiment["dropout"]
    )

    optimizer = torch.optim.Adam(
        [
            p for p in model.parameters()
            if p.requires_grad
        ],
        lr=learning_rate,
        weight_decay=experiment["weight_decay"]
    )

    # --------------------------------------------------------
    # 2. W&B
    # --------------------------------------------------------

    run_name = (
        f"pairwise_robust_{name}_seed{seed}"
    )

    run = wandb.init(
        entity="kimbortis-hochschule-luzern",
        project="nlp-art-hs26-project1",
        group="pairwise-robustness",
        job_type="regularization-comparison",
        name=run_name,
        config={
            "architecture": "PairwiseGRU",
            "hidden_dim": 128,
            "num_layers": 2,
            "dropout": experiment["dropout"],
            "learning_rate": learning_rate,
            "weight_decay": experiment["weight_decay"],
            "batch_size": 32,
            "seed": seed,
            "max_epochs": max_epochs,
            "patience": patience,
            "optimizer": "Adam",
            "embedding_frozen": True,
            "development_vocabulary_size": len(dev_vocab),
            "training_examples": len(robust_train_data),
            "checkpoint_selection": "normal_dev_loss",
            "device": str(device)
        }
    )

    run.define_metric("epoch")
    run.define_metric("train/*", step_metric="epoch")
    run.define_metric("normal_dev/*", step_metric="epoch")

    # Save both checkpoint types
    loss_checkpoint = (
        checkpoint_dir /
        f"{run_name}_{run.id}_best_loss.pt"
    )

    accuracy_checkpoint = (
        checkpoint_dir /
        f"{run_name}_{run.id}_best_accuracy.pt"
    )

    best_loss = float("inf")
    best_accuracy = -1.0
    best_accuracy_loss = float("inf")

    best_loss_epoch = 0
    best_accuracy_epoch = 0

    no_improvement = 0

    # Same RNG convention as previous experiments
    random.seed(seed + 1)
    np.random.seed(seed + 1)
    torch.manual_seed(seed + 1)
    torch.cuda.manual_seed_all(seed + 1)

    try:
        # ----------------------------------------------------
        # 3. Training
        # ----------------------------------------------------

        for epoch in range(1, max_epochs + 1):

            train_metrics = train_one_epoch(
                model,
                robust_train_loader,
                optimizer,
                criterion,
                device
            )

            # Normal development holdout ONLY
            normal_metrics = evaluate(
                model,
                normal_dev_loader,
                criterion,
                device
            )

            val_loss = normal_metrics["loss"]
            val_accuracy = normal_metrics["accuracy"]

            run.log({
                "epoch": epoch,
                "train/loss": train_metrics["loss"],
                "train/accuracy": train_metrics["accuracy"],
                "normal_dev/loss": val_loss,
                "normal_dev/accuracy": val_accuracy
            })

            # -----------------------------------------------
            # A. Lowest validation-loss checkpoint
            # -----------------------------------------------

            if val_loss < best_loss:

                best_loss = val_loss
                best_loss_epoch = epoch
                no_improvement = 0

                torch.save(
                    model.state_dict(),
                    loss_checkpoint
                )

            else:
                no_improvement += 1

            # -----------------------------------------------
            # B. Highest validation-accuracy checkpoint
            # -----------------------------------------------

            if (
                val_accuracy > best_accuracy
                or (
                    val_accuracy == best_accuracy
                    and val_loss < best_accuracy_loss
                )
            ):

                best_accuracy = val_accuracy
                best_accuracy_loss = val_loss
                best_accuracy_epoch = epoch

                torch.save(
                    model.state_dict(),
                    accuracy_checkpoint
                )

            if (
                epoch == 1
                or epoch % 5 == 0
                or epoch == best_loss_epoch
            ):
                print(
                    f"Epoch {epoch:02d} | "
                    f"Train: {train_metrics['accuracy']:.2%} | "
                    f"Normal dev: {val_accuracy:.2%} | "
                    f"Normal loss: {val_loss:.4f}"
                )

            # Early stopping based on validation LOSS
            if no_improvement >= patience:
                print(
                    f"Early stopping at epoch {epoch}"
                )
                break

        # ----------------------------------------------------
        # 4. Evaluate both saved checkpoint types
        # ----------------------------------------------------

        checkpoint_results = {}

        for selection, path in [
            ("loss", loss_checkpoint),
            ("accuracy", accuracy_checkpoint)
        ]:

            model.load_state_dict(
                torch.load(
                    path,
                    map_location=device,
                    weights_only=True
                )
            )

            model.eval()

            # Normal development holdout
            normal_result = evaluate(
                model,
                normal_dev_loader,
                criterion,
                device
            )

            # Hard holdout: evaluated only AFTER training
            hard_result = evaluate(
                model,
                hard_dev_loader,
                criterion,
                device
            )

            checkpoint_results[selection] = {
                "normal_accuracy": normal_result["accuracy"],
                "normal_loss": normal_result["loss"],
                "hard_accuracy": hard_result["accuracy"],
                "hard_loss": hard_result["loss"],
                "checkpoint": str(path)
            }

            print(
                f"\n=== {name.upper()} | "
                f"{selection.upper()} CHECKPOINT ==="
            )

            print(
                f"Normal accuracy: "
                f"{normal_result['accuracy']:.2%}"
            )

            print(
                f"Normal loss: "
                f"{normal_result['loss']:.4f}"
            )

            print(
                f"Hard accuracy: "
                f"{hard_result['accuracy']:.2%}"
            )

            print(
                f"Hard loss: "
                f"{hard_result['loss']:.4f}"
            )

        # Validate checkpoint restoration
        assert abs(
            checkpoint_results["loss"]["normal_loss"]
            - best_loss
        ) < 1e-5

        assert abs(
            checkpoint_results["accuracy"]["normal_accuracy"]
            - best_accuracy
        ) < 1e-6

        # ----------------------------------------------------
        # 5. Store results
        # ----------------------------------------------------

        run.summary["best_loss_epoch"] = best_loss_epoch
        run.summary["best_accuracy_epoch"] = best_accuracy_epoch
        run.summary["best_normal_dev_loss"] = best_loss
        run.summary["best_normal_dev_accuracy"] = best_accuracy

        for selection, result in checkpoint_results.items():
            for metric, value in result.items():
                if metric != "checkpoint":
                    run.summary[
                        f"{selection}/{metric}"
                    ] = value

        robustness_results.append({
            "name": name,
            "dropout": experiment["dropout"],
            "weight_decay": experiment["weight_decay"],
            "best_loss_epoch": best_loss_epoch,
            "best_accuracy_epoch": best_accuracy_epoch,
            "results": checkpoint_results
        })

        print("\nCheckpoint verification: PASS")

    finally:
        run.finish()

        del model, optimizer

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ============================================================
# Final comparison
# ============================================================

print("\n=== ROBUSTNESS EXPERIMENT COMPARISON ===")

for result in robustness_results:

    print(f"\n{result['name'].upper()}")

    print(
        "Best loss epoch:",
        result["best_loss_epoch"]
    )

    print(
        "Best accuracy epoch:",
        result["best_accuracy_epoch"]
    )

    for selection in ["loss", "accuracy"]:

        metrics = result["results"][selection]

        print(
            f"{selection.upper():8s} | "
            f"Normal: {metrics['normal_accuracy']:.2%} | "
            f"Hard: {metrics['hard_accuracy']:.2%} | "
            f"Normal loss: {metrics['normal_loss']:.4f}"
        )

print("\nExperiment 6: COMPLETE")
print("Official test set: NOT ACCESSED")



ROBUSTNESS EXPERIMENT: BASELINE


Epoch 01 | Train: 62.62% | Normal dev: 66.07% | Normal loss: 0.6087
Epoch 02 | Train: 67.13% | Normal dev: 67.14% | Normal loss: 0.5999
Epoch 03 | Train: 68.26% | Normal dev: 68.33% | Normal loss: 0.5924
Epoch 04 | Train: 69.55% | Normal dev: 68.21% | Normal loss: 0.5869
Epoch 05 | Train: 70.46% | Normal dev: 68.57% | Normal loss: 0.5797
Epoch 06 | Train: 72.05% | Normal dev: 68.57% | Normal loss: 0.5707
Epoch 10 | Train: 78.65% | Normal dev: 69.29% | Normal loss: 0.5923
Epoch 15 | Train: 90.03% | Normal dev: 69.52% | Normal loss: 0.8490
Epoch 20 | Train: 96.89% | Normal dev: 71.55% | Normal loss: 1.7506
Early stopping at epoch 21

=== BASELINE | LOSS CHECKPOINT ===
Normal accuracy: 68.57%
Normal loss: 0.5707
Hard accuracy: 66.55%
Hard loss: 0.6733

=== BASELINE | ACCURACY CHECKPOINT ===
Normal accuracy: 71.55%
Normal loss: 1.7506
Hard accuracy: 66.07%
Hard loss: 3.6993

Checkpoint verification: PASS


epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
normal_dev/accuracy,▁▂▄▄▄▄▅▄▅▅▇▄▆▆▅▆▆▆▆█▅
normal_dev/loss,▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▄▄▅▅▆█
train/accuracy,▁▂▂▂▃▃▃▃▄▄▅▅▆▆▆▇▇████
train/loss,█▇▇▇▇▇▆▆▆▅▅▅▄▃▃▂▂▂▁▁▁
accuracy/hard_accuracy,0.66071
accuracy/hard_loss,3.69927
accuracy/normal_accuracy,0.71548
accuracy/normal_loss,1.7506
best_accuracy_epoch,20
best_loss_epoch,6



ROBUSTNESS EXPERIMENT: REGULARIZED


Epoch 01 | Train: 50.24% | Normal dev: 49.29% | Normal loss: 0.6931
Epoch 02 | Train: 50.04% | Normal dev: 49.17% | Normal loss: 0.6931
Epoch 03 | Train: 49.92% | Normal dev: 50.24% | Normal loss: 0.6931
Epoch 04 | Train: 50.06% | Normal dev: 50.12% | Normal loss: 0.6931
Epoch 05 | Train: 50.54% | Normal dev: 50.48% | Normal loss: 0.6931
Epoch 10 | Train: 49.93% | Normal dev: 50.60% | Normal loss: 0.6931
Epoch 15 | Train: 50.00% | Normal dev: 48.21% | Normal loss: 0.6931
Epoch 20 | Train: 50.20% | Normal dev: 53.45% | Normal loss: 0.6931
Early stopping at epoch 20

=== REGULARIZED | LOSS CHECKPOINT ===
Normal accuracy: 50.48%
Normal loss: 0.6931
Hard accuracy: 49.64%
Hard loss: 0.6931

=== REGULARIZED | ACCURACY CHECKPOINT ===
Normal accuracy: 53.45%
Normal loss: 0.6931
Hard accuracy: 45.48%
Hard loss: 0.6931

Checkpoint verification: PASS


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
normal_dev/accuracy,▃▃▄▄▅▃▁▄▃▅▃▄▄▅▂▄▆▃▃█
normal_dev/loss,█▃▂▁▁▂▂▂▂▂▂▂▃▂▂▂▂▂▃▇
train/accuracy,▆▅▅▅█▄▄▅▁▅▆█▆█▅▄▆▆▆▆
train/loss,█▅▅▄▂▄▄▃▄▃▃▁▄▅▄▂▂▂▂▇
accuracy/hard_accuracy,0.45476
accuracy/hard_loss,0.69315
accuracy/normal_accuracy,0.53452
accuracy/normal_loss,0.69315
best_accuracy_epoch,20
best_loss_epoch,5



=== ROBUSTNESS EXPERIMENT COMPARISON ===

BASELINE
Best loss epoch: 6
Best accuracy epoch: 20
LOSS     | Normal: 68.57% | Hard: 66.55% | Normal loss: 0.5707
ACCURACY | Normal: 71.55% | Hard: 66.07% | Normal loss: 1.7506

REGULARIZED
Best loss epoch: 5
Best accuracy epoch: 20
LOSS     | Normal: 50.48% | Hard: 49.64% | Normal loss: 0.6931
ACCURACY | Normal: 53.45% | Hard: 45.48% | Normal loss: 0.6931

Experiment 6: COMPLETE
Official test set: NOT ACCESSED


In [12]:

batch = next(iter(normal_dev_loader))

print("=== PAIRWISE GRU PREDICTION DIAGNOSTIC ===")

for result in robustness_results:

    name = result["name"]

    # Restore lowest-loss checkpoint
    model = create_robust_pairwise_model(
        hidden_dim=128,
        dropout=result["dropout"]
    )

    checkpoint = result["results"]["loss"]["checkpoint"]

    model.load_state_dict(
        torch.load(
            checkpoint,
            map_location=device,
            weights_only=True
        )
    )

    model.eval()

    with torch.inference_mode():
        logits = model(
            batch["input_ids"].to(device),
            lengths=batch["lengths"]
        )

        probabilities = torch.softmax(logits, dim=1)
        margins = logits[:, 1] - logits[:, 0]
        predictions = logits.argmax(dim=1)

    print(f"\n{name.upper()}")
    print(
        "Mean absolute score difference:",
        f"{margins.abs().mean().item():.8f}"
    )
    print(
        "Maximum absolute score difference:",
        f"{margins.abs().max().item():.8f}"
    )
    print(
        "Mean prediction confidence:",
        f"{probabilities.max(dim=1).values.mean().item():.4f}"
    )
    print(
        "Predicted H2:",
        f"{(predictions == 1).float().mean().item():.2%}"
    )
    print("First five logits:")
    print(logits[:5].cpu())

    del model

print("\nDiagnostic: COMPLETE")
print("Official test set: NOT ACCESSED")


=== PAIRWISE GRU PREDICTION DIAGNOSTIC ===

BASELINE
Mean absolute score difference: 0.89108133
Maximum absolute score difference: 4.66181231
Mean prediction confidence: 0.6800
Predicted H2: 46.88%
First five logits:
tensor([[ -9.6441, -10.1869],
        [-12.8750, -13.8973],
        [ -5.0754,  -9.7372],
        [ -9.3369, -10.0645],
        [-11.8514, -11.8382]])

REGULARIZED
Mean absolute score difference: 0.00000057
Maximum absolute score difference: 0.00000651
Mean prediction confidence: 0.5000
Predicted H2: 50.00%
First five logits:
tensor([[-0.0061, -0.0061],
        [-0.0075, -0.0075],
        [-0.0060, -0.0060],
        [-0.0073, -0.0073],
        [-0.0061, -0.0061]])

Diagnostic: COMPLETE
Official test set: NOT ACCESSED


In [13]:

import random
import numpy as np
import torch
import torch.nn as nn
import wandb

experiments = [
    {
        "name": "dropout_only",
        "dropout": 0.3,
        "weight_decay": 0.0
    },
    {
        "name": "weight_decay_only",
        "dropout": 0.0,
        "weight_decay": 0.0001
    }
]

seed = 42
learning_rate = 0.001
max_epochs = 60
patience = 15

checkpoint_dir = (
    repo_root / "project1" / "artifacts"
)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

ablation_results = []

criterion = nn.CrossEntropyLoss()

# ============================================================
# Train each configuration
# ============================================================

for experiment in experiments:

    name = experiment["name"]

    print(f"\n{'=' * 60}")
    print(f"ROBUSTNESS EXPERIMENT: {name.upper()}")
    print(f"{'=' * 60}")

    # --------------------------------------------------------
    # 1. Reproducibility
    # --------------------------------------------------------

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Fresh model
    model = create_robust_pairwise_model(
        hidden_dim=128,
        dropout=experiment["dropout"]
    )

    optimizer = torch.optim.Adam(
        [
            p for p in model.parameters()
            if p.requires_grad
        ],
        lr=learning_rate,
        weight_decay=experiment["weight_decay"]
    )

    # --------------------------------------------------------
    # 2. W&B
    # --------------------------------------------------------

    run_name = (
        f"pairwise_ablation_{name}_seed{seed}"
    )

    run = wandb.init(
        entity="kimbortis-hochschule-luzern",
        project="nlp-art-hs26-project1",
        group="pairwise-regularization-ablation",
        job_type="regularization-ablation",
        name=run_name,
        config={
            "architecture": "PairwiseGRU",
            "hidden_dim": 128,
            "num_layers": 2,
            "dropout": experiment["dropout"],
            "learning_rate": learning_rate,
            "weight_decay": experiment["weight_decay"],
            "batch_size": 32,
            "seed": seed,
            "max_epochs": max_epochs,
            "patience": patience,
            "optimizer": "Adam",
            "embedding_frozen": True,
            "development_vocabulary_size": len(dev_vocab),
            "training_examples": len(robust_train_data),
            "checkpoint_selection": "normal_dev_loss",
            "device": str(device)
        }
    )

    run.define_metric("epoch")
    run.define_metric("train/*", step_metric="epoch")
    run.define_metric("normal_dev/*", step_metric="epoch")

    # Save both checkpoint types
    loss_checkpoint = (
        checkpoint_dir /
        f"{run_name}_{run.id}_best_loss.pt"
    )

    accuracy_checkpoint = (
        checkpoint_dir /
        f"{run_name}_{run.id}_best_accuracy.pt"
    )

    best_loss = float("inf")
    best_accuracy = -1.0
    best_accuracy_loss = float("inf")

    best_loss_epoch = 0
    best_accuracy_epoch = 0

    no_improvement = 0

    # Same RNG convention as previous experiments
    random.seed(seed + 1)
    np.random.seed(seed + 1)
    torch.manual_seed(seed + 1)
    torch.cuda.manual_seed_all(seed + 1)

    try:
        # ----------------------------------------------------
        # 3. Training
        # ----------------------------------------------------

        for epoch in range(1, max_epochs + 1):

            train_metrics = train_one_epoch(
                model,
                robust_train_loader,
                optimizer,
                criterion,
                device
            )

            # Normal development holdout ONLY
            normal_metrics = evaluate(
                model,
                normal_dev_loader,
                criterion,
                device
            )

            val_loss = normal_metrics["loss"]
            val_accuracy = normal_metrics["accuracy"]

            run.log({
                "epoch": epoch,
                "train/loss": train_metrics["loss"],
                "train/accuracy": train_metrics["accuracy"],
                "normal_dev/loss": val_loss,
                "normal_dev/accuracy": val_accuracy
            })

            # -----------------------------------------------
            # A. Lowest validation-loss checkpoint
            # -----------------------------------------------

            if val_loss < best_loss:

                best_loss = val_loss
                best_loss_epoch = epoch
                no_improvement = 0

                torch.save(
                    model.state_dict(),
                    loss_checkpoint
                )

            else:
                no_improvement += 1

            # -----------------------------------------------
            # B. Highest validation-accuracy checkpoint
            # -----------------------------------------------

            if (
                val_accuracy > best_accuracy
                or (
                    val_accuracy == best_accuracy
                    and val_loss < best_accuracy_loss
                )
            ):

                best_accuracy = val_accuracy
                best_accuracy_loss = val_loss
                best_accuracy_epoch = epoch

                torch.save(
                    model.state_dict(),
                    accuracy_checkpoint
                )

            if (
                epoch == 1
                or epoch % 5 == 0
                or epoch == best_loss_epoch
            ):
                print(
                    f"Epoch {epoch:02d} | "
                    f"Train: {train_metrics['accuracy']:.2%} | "
                    f"Normal dev: {val_accuracy:.2%} | "
                    f"Normal loss: {val_loss:.4f}"
                )

            # Early stopping based on validation LOSS
            if no_improvement >= patience:
                print(
                    f"Early stopping at epoch {epoch}"
                )
                break

        # ----------------------------------------------------
        # 4. Evaluate both saved checkpoint types
        # ----------------------------------------------------

        checkpoint_results = {}

        for selection, path in [
            ("loss", loss_checkpoint),
            ("accuracy", accuracy_checkpoint)
        ]:

            model.load_state_dict(
                torch.load(
                    path,
                    map_location=device,
                    weights_only=True
                )
            )

            model.eval()

            # Normal development holdout
            normal_result = evaluate(
                model,
                normal_dev_loader,
                criterion,
                device
            )

            # Hard holdout: evaluated only AFTER training
            hard_result = evaluate(
                model,
                hard_dev_loader,
                criterion,
                device
            )

            checkpoint_results[selection] = {
                "normal_accuracy": normal_result["accuracy"],
                "normal_loss": normal_result["loss"],
                "hard_accuracy": hard_result["accuracy"],
                "hard_loss": hard_result["loss"],
                "checkpoint": str(path)
            }

            print(
                f"\n=== {name.upper()} | "
                f"{selection.upper()} CHECKPOINT ==="
            )

            print(
                f"Normal accuracy: "
                f"{normal_result['accuracy']:.2%}"
            )

            print(
                f"Normal loss: "
                f"{normal_result['loss']:.4f}"
            )

            print(
                f"Hard accuracy: "
                f"{hard_result['accuracy']:.2%}"
            )

            print(
                f"Hard loss: "
                f"{hard_result['loss']:.4f}"
            )

        # Validate checkpoint restoration
        assert abs(
            checkpoint_results["loss"]["normal_loss"]
            - best_loss
        ) < 1e-5

        assert abs(
            checkpoint_results["accuracy"]["normal_accuracy"]
            - best_accuracy
        ) < 1e-6

        # ----------------------------------------------------
        # 5. Store results
        # ----------------------------------------------------

        run.summary["best_loss_epoch"] = best_loss_epoch
        run.summary["best_accuracy_epoch"] = best_accuracy_epoch
        run.summary["best_normal_dev_loss"] = best_loss
        run.summary["best_normal_dev_accuracy"] = best_accuracy

        for selection, result in checkpoint_results.items():
            for metric, value in result.items():
                if metric != "checkpoint":
                    run.summary[
                        f"{selection}/{metric}"
                    ] = value

        ablation_results.append({
            "name": name,
            "dropout": experiment["dropout"],
            "weight_decay": experiment["weight_decay"],
            "best_loss_epoch": best_loss_epoch,
            "best_accuracy_epoch": best_accuracy_epoch,
            "results": checkpoint_results
        })

        print("\nCheckpoint verification: PASS")

    finally:
        run.finish()

        del model, optimizer

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ============================================================
# Final comparison
# ============================================================

print("\n=== ROBUSTNESS EXPERIMENT COMPARISON ===")

for result in ablation_results:

    print(f"\n{result['name'].upper()}")

    print(
        "Best loss epoch:",
        result["best_loss_epoch"]
    )

    print(
        "Best accuracy epoch:",
        result["best_accuracy_epoch"]
    )

    for selection in ["loss", "accuracy"]:

        metrics = result["results"][selection]

        print(
            f"{selection.upper():8s} | "
            f"Normal: {metrics['normal_accuracy']:.2%} | "
            f"Hard: {metrics['hard_accuracy']:.2%} | "
            f"Normal loss: {metrics['normal_loss']:.4f}"
        )

print("\nExperiment 6: COMPLETE")
print("Official test set: NOT ACCESSED")



ROBUSTNESS EXPERIMENT: DROPOUT_ONLY


Epoch 01 | Train: 50.33% | Normal dev: 53.93% | Normal loss: 0.6931
Epoch 05 | Train: 50.04% | Normal dev: 49.52% | Normal loss: 0.6931
Epoch 06 | Train: 50.22% | Normal dev: 58.57% | Normal loss: 0.6922
Epoch 10 | Train: 50.25% | Normal dev: 52.14% | Normal loss: 0.6931
Epoch 15 | Train: 50.06% | Normal dev: 51.19% | Normal loss: 0.6931
Epoch 20 | Train: 50.33% | Normal dev: 52.14% | Normal loss: 0.6931
Early stopping at epoch 21

=== DROPOUT_ONLY | LOSS CHECKPOINT ===
Normal accuracy: 58.57%
Normal loss: 0.6922
Hard accuracy: 58.10%
Hard loss: 0.6911

=== DROPOUT_ONLY | ACCURACY CHECKPOINT ===
Normal accuracy: 58.57%
Normal loss: 0.6922
Hard accuracy: 58.10%
Hard loss: 0.6911

Checkpoint verification: PASS


epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
normal_dev/accuracy,▆▅▅▅▄█▄▅▅▅▅▅▅▁▄▅▅▅▅▅▅
normal_dev/loss,█████▁███████████████
train/accuracy,▆▇▂▁▄▅▇▄▅▅▆▆▆▆▅▅█▅▆▆▆
train/loss,▂▃▃▂▂▃█▁▁▁▁▁▁▁▅▁▁▁▁▁▁
accuracy/hard_accuracy,0.58095
accuracy/hard_loss,0.69107
accuracy/normal_accuracy,0.58571
accuracy/normal_loss,0.69223
best_accuracy_epoch,6
best_loss_epoch,6



ROBUSTNESS EXPERIMENT: WEIGHT_DECAY_ONLY


Epoch 01 | Train: 50.00% | Normal dev: 52.14% | Normal loss: 0.6931
Epoch 05 | Train: 50.33% | Normal dev: 52.14% | Normal loss: 0.6931
Epoch 10 | Train: 50.33% | Normal dev: 52.14% | Normal loss: 0.6931
Epoch 15 | Train: 50.33% | Normal dev: 52.14% | Normal loss: 0.6931
Early stopping at epoch 16

=== WEIGHT_DECAY_ONLY | LOSS CHECKPOINT ===
Normal accuracy: 52.14%
Normal loss: 0.6931
Hard accuracy: 49.17%
Hard loss: 0.6931

=== WEIGHT_DECAY_ONLY | ACCURACY CHECKPOINT ===
Normal accuracy: 52.14%
Normal loss: 0.6931
Hard accuracy: 49.17%
Hard loss: 0.6931

Checkpoint verification: PASS


epoch,▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
normal_dev/accuracy,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
normal_dev/loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/accuracy,▁███████████████
train/loss,▁███████████████
accuracy/hard_accuracy,0.49167
accuracy/hard_loss,0.69315
accuracy/normal_accuracy,0.52143
accuracy/normal_loss,0.69315
best_accuracy_epoch,1
best_loss_epoch,1



=== ROBUSTNESS EXPERIMENT COMPARISON ===

DROPOUT_ONLY
Best loss epoch: 6
Best accuracy epoch: 6
LOSS     | Normal: 58.57% | Hard: 58.10% | Normal loss: 0.6922
ACCURACY | Normal: 58.57% | Hard: 58.10% | Normal loss: 0.6922

WEIGHT_DECAY_ONLY
Best loss epoch: 1
Best accuracy epoch: 1
LOSS     | Normal: 52.14% | Hard: 49.17% | Normal loss: 0.6931
ACCURACY | Normal: 52.14% | Hard: 49.17% | Normal loss: 0.6931

Experiment 6: COMPLETE
Official test set: NOT ACCESSED


### Regularization Results

Results using the lowest-validation-loss checkpoints:

| Configuration | Normal holdout | Hard holdout |
|---|---:|---:|
| Baseline | 68.57% | 66.55% |
| Dropout + weight decay | 50.48% | 49.64% |
| Dropout only | 58.57% | 58.10% |
| Weight decay only | 52.14% | 49.17% |

**Findings:**

- The unregularized baseline achieved the highest accuracy on both holdouts.
- Combined regularization caused the model to produce nearly identical scores for both hypotheses, resulting in approximately chance-level performance.
- Separate dropout and weight decay experiments also performed substantially worse than the baseline.
- The baseline's highest-accuracy checkpoint reached 71.55% on the normal holdout but had much higher validation loss (1.7506), indicating severe overconfidence.

**Conclusion:** These regularization settings did not improve generalization. The baseline performed best, although its performance decreased on low-similarity examples.

All results are from development data, not the official test set.

## Context Sensitivity and Hypothesis-Only Diagnostics

We investigate whether the PairwiseGRU uses contextual observations or primarily relies on hypothesis-level patterns.

We evaluate shuffled observations, repeat the ablation across 10 permutations, and train a hypothesis-only baseline.

These experiments use training and validation data only, without accessing the official test set.

In [14]:

import numpy as np
import torch

# Validation-only observation ablation
# No training. No test data.

rng = np.random.default_rng(2026)
n = len(val_data)
indices = np.arange(n)

# Create a permutation with no unchanged positions
donors = rng.permutation(n)
while np.any(donors == indices):
    donors = rng.permutation(n)

def replace_observations(example, idx):
    donor = val_data[int(donors[idx])]
    return {
        "observation_1": donor["observation_1"],
        "observation_2": donor["observation_2"],
    }

ablated_data = val_data.map(
    replace_observations,
    with_indices=True,
    load_from_cache_file=False
)

# Hypotheses and original labels stay unchanged
for field in ["hypothesis_1", "hypothesis_2", "label"]:
    assert list(ablated_data[field]) == list(val_data[field])

ablated_loader = create_dataloader(
    ablated_data,
    vocab,
    batch_size=32,
    shuffle=False
)

original_logits, labels = collect_outputs(
    diagnostic_model, val_loader
)

ablated_logits, ablated_labels = collect_outputs(
    diagnostic_model, ablated_loader
)

assert torch.equal(labels, ablated_labels)

original_pred = original_logits.argmax(dim=1)
ablated_pred = ablated_logits.argmax(dim=1)

original_accuracy = (
    original_pred == labels
).float().mean().item()

ablated_accuracy = (
    ablated_pred == labels
).float().mean().item()

prediction_change = (
    original_pred != ablated_pred
).float().mean().item()

original_margin = original_logits[:, 0] - original_logits[:, 1]
ablated_margin = ablated_logits[:, 0] - ablated_logits[:, 1]

margin_change = (
    original_margin - ablated_margin
).abs().mean().item()

print("=== VALIDATION CONTEXT ABLATION ===")
print(f"Original accuracy: {original_accuracy:.2%}")
print(f"Shuffled-context accuracy: {ablated_accuracy:.2%}")
print(f"Accuracy difference: {100 * (ablated_accuracy - original_accuracy):+.2f} pp")
print(f"Prediction change rate: {prediction_change:.2%}")
print(f"Mean absolute margin change: {margin_change:.4f}")
print("Official test set: NOT ACCESSED")


Map: 100%|██████████| 1000/1000 [00:00<00:00, 5053.73 examples/s]


=== VALIDATION CONTEXT ABLATION ===
Original accuracy: 71.10%
Shuffled-context accuracy: 67.40%
Accuracy difference: -3.70 pp
Prediction change rate: 14.90%
Mean absolute margin change: 0.8508
Official test set: NOT ACCESSED


In [15]:

# Validation only — no training or test access

def ablate_observations(example, idx, mode):
    donor = val_data[int(donors[idx])]

    updates = {}

    if mode in ("O1", "both"):
        updates["observation_1"] = donor["observation_1"]

    if mode in ("O2", "both"):
        updates["observation_2"] = donor["observation_2"]

    return updates


original_predictions = original_logits.argmax(dim=1)
original_accuracy = (
    original_predictions == labels
).float().mean().item()

original_margin = original_logits[:, 0] - original_logits[:, 1]

print("=== OBSERVATION SENSITIVITY ===")

for mode in ("O1", "O2", "both"):

    modified_data = val_data.map(
        lambda example, idx: ablate_observations(
            example, idx, mode
        ),
        with_indices=True,
        load_from_cache_file=False
    )

    loader = create_dataloader(
        modified_data,
        vocab,
        batch_size=32,
        shuffle=False
    )

    logits, modified_labels = collect_outputs(
        diagnostic_model, loader
    )

    assert torch.equal(labels, modified_labels)

    predictions = logits.argmax(dim=1)

    accuracy = (
        predictions == labels
    ).float().mean().item()

    flip_rate = (
        predictions != original_predictions
    ).float().mean().item()

    margin = logits[:, 0] - logits[:, 1]

    margin_change = (
        margin - original_margin
    ).abs().mean().item()

    print(f"\nShuffle: {mode}")
    print(f"Accuracy: {accuracy:.2%}")
    print(
        f"Accuracy change: "
        f"{100 * (accuracy - original_accuracy):+.2f} pp"
    )
    print(f"Prediction changes: {flip_rate:.2%}")
    print(f"Mean margin change: {margin_change:.4f}")

print("\nOfficial test set: NOT ACCESSED")


=== OBSERVATION SENSITIVITY ===


Map: 100%|██████████| 1000/1000 [00:00<00:00, 6033.21 examples/s]



Shuffle: O1
Accuracy: 69.40%
Accuracy change: -1.70 pp
Prediction changes: 5.50%
Mean margin change: 0.3800


Map: 100%|██████████| 1000/1000 [00:00<00:00, 8328.87 examples/s]



Shuffle: O2
Accuracy: 68.50%
Accuracy change: -2.60 pp
Prediction changes: 12.40%
Mean margin change: 0.7554


Map: 100%|██████████| 1000/1000 [00:00<00:00, 9934.33 examples/s]



Shuffle: both
Accuracy: 67.40%
Accuracy change: -3.70 pp
Prediction changes: 14.90%
Mean margin change: 0.8508

Official test set: NOT ACCESSED


In [16]:

# Repeated validation-only context sensitivity
# No training and no test-set access

import numpy as np
import torch

rng = np.random.default_rng(2027)
n = len(val_data)
original_pred = original_logits.argmax(dim=1)
original_margin = (
    original_logits[:, 0] - original_logits[:, 1]
)

results = []

for repeat in range(10):
    donors = rng.permutation(n)

    # Ensure observations come from other examples
    while np.any(donors == np.arange(n)):
        donors = rng.permutation(n)

    for mode in ("O1", "O2", "both"):
        def replace_context(example, idx):
            donor = val_data[int(donors[idx])]
            updates = {}

            if mode in ("O1", "both"):
                updates["observation_1"] = donor["observation_1"]

            if mode in ("O2", "both"):
                updates["observation_2"] = donor["observation_2"]

            return updates

        modified = val_data.map(
            replace_context,
            with_indices=True,
            load_from_cache_file=False
        )

        loader = create_dataloader(
            modified, vocab,
            batch_size=32,
            shuffle=False
        )

        logits, current_labels = collect_outputs(
            diagnostic_model, loader
        )

        assert torch.equal(labels, current_labels)

        predictions = logits.argmax(dim=1)
        margins = logits[:, 0] - logits[:, 1]

        results.append({
            "mode": mode,
            "flip_rate": (
                (predictions != original_pred)
                .float().mean().item() * 100
            ),
            "margin_change": (
                (margins - original_margin)
                .abs().mean().item()
            )
        })

import pandas as pd

summary = (
    pd.DataFrame(results)
    .groupby("mode")
    .agg(
        flip_mean=("flip_rate", "mean"),
        flip_std=("flip_rate", "std"),
        margin_mean=("margin_change", "mean"),
        margin_std=("margin_change", "std")
    )
)

print(summary.round(3))
print("\nOfficial test set: NOT ACCESSED")


Map: 100%|██████████| 1000/1000 [00:00<00:00, 8023.86 examples/s]


      flip_mean  flip_std  margin_mean  margin_std
mode                                              
O1         4.80     0.389        0.377       0.010
O2        11.99     0.689        0.742       0.026
both      13.11     0.666        0.856       0.023

Official test set: NOT ACCESSED


In [17]:

# Hypothesis-only diagnostic
# Train + validation only. Never access test.

import copy
import random
import numpy as np
import torch
import torch.nn as nn

def hide_observations(example):
    return {
        "observation_1": "the",
        "observation_2": "the"
    }

assert "the" in vocab

hyp_train = train_data.map(
    hide_observations,
    load_from_cache_file=False
)

hyp_val = val_data.map(
    hide_observations,
    load_from_cache_file=False
)

hyp_train_loader = create_dataloader(
    hyp_train, vocab, batch_size=32, shuffle=True
)

hyp_val_loader = create_dataloader(
    hyp_val, vocab, batch_size=32, shuffle=False
)

# Same architecture and hyperparameters
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

model = create_pairwise_model(
    hidden_dim=128,
    dropout=0.0
)

optimizer = torch.optim.Adam(
    (p for p in model.parameters() if p.requires_grad),
    lr=0.001,
    weight_decay=0.0
)

criterion = nn.CrossEntropyLoss()

best_accuracy = -1.0
best_loss = float("inf")
best_state = None
patience_counter = 0

for epoch in range(1, 61):
    train_metrics = train_one_epoch(
        model, hyp_train_loader,
        optimizer, criterion, device
    )

    val_metrics = evaluate(
        model, hyp_val_loader,
        criterion, device
    )

    # Match original checkpoint-selection rule
    improved = (
        val_metrics["accuracy"] > best_accuracy
        or (
            val_metrics["accuracy"] == best_accuracy
            and val_metrics["loss"] < best_loss
        )
    )

    if improved:
        best_accuracy = val_metrics["accuracy"]
        best_loss = val_metrics["loss"]
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:02d} | "
            f"Train {train_metrics['accuracy']:.2%} | "
            f"Val {val_metrics['accuracy']:.2%}"
        )

    if patience_counter >= 15:
        break

model.load_state_dict(best_state)

print("\n=== HYPOTHESIS-ONLY BASELINE ===")
print(f"Validation accuracy: {best_accuracy:.2%}")
print("Original PairwiseGRU: 71.10%")
print(f"Validation loss: {best_loss:.4f}")
print("Official test set: NOT ACCESSED")


Map: 100%|██████████| 1000/1000 [00:00<00:00, 13295.08 examples/s]


Epoch 01 | Train 63.55% | Val 63.60%
Epoch 05 | Train 70.59% | Val 68.00%
Epoch 10 | Train 76.63% | Val 67.10%
Epoch 15 | Train 86.02% | Val 68.40%
Epoch 20 | Train 94.59% | Val 65.40%

=== HYPOTHESIS-ONLY BASELINE ===
Validation accuracy: 69.10%
Original PairwiseGRU: 71.10%
Validation loss: 0.6064
Official test set: NOT ACCESSED


In [18]:

# Paired error analysis — validation only
# No training or test access

import numpy as np
import torch

full_logits, full_labels = collect_outputs(
    diagnostic_model, val_loader
)

hyp_logits, hyp_labels = collect_outputs(
    model, hyp_val_loader
)

assert torch.equal(full_labels, hyp_labels)

full_pred = full_logits.argmax(dim=1)
hyp_pred = hyp_logits.argmax(dim=1)

full_correct = full_pred == full_labels
hyp_correct = hyp_pred == full_labels

both_correct = (full_correct & hyp_correct).sum().item()
full_only = (full_correct & ~hyp_correct).sum().item()
hyp_only = (~full_correct & hyp_correct).sum().item()
both_wrong = (~full_correct & ~hyp_correct).sum().item()

agreement = (
    full_pred == hyp_pred
).float().mean().item()

# Paired bootstrap for difference in validation accuracy
differences = (
    full_correct.int() - hyp_correct.int()
).cpu().numpy()

rng = np.random.default_rng(42)
resamples = rng.integers(
    0, len(differences),
    size=(3000, len(differences))
)

boot_deltas = differences[resamples].mean(axis=1) * 100
ci_low, ci_high = np.percentile(boot_deltas, [2.5, 97.5])

print("=== PAIRED VALIDATION ANALYSIS ===")
print(f"Both correct: {both_correct}")
print(f"Full only correct: {full_only}")
print(f"Hypothesis only correct: {hyp_only}")
print(f"Both wrong: {both_wrong}")
print(f"Prediction agreement: {agreement:.2%}")
print(f"Accuracy difference: {differences.mean()*100:+.2f} pp")
print(f"95% bootstrap interval: [{ci_low:.2f}, {ci_high:.2f}] pp")
print("Official test set: NOT ACCESSED")


=== PAIRED VALIDATION ANALYSIS ===
Both correct: 606
Full only correct: 105
Hypothesis only correct: 85
Both wrong: 204
Prediction agreement: 81.00%
Accuracy difference: +2.00 pp
95% bootstrap interval: [-0.70, 4.70] pp
Official test set: NOT ACCESSED


### Results and Interpretation

| Experiment | Validation accuracy |
|---|---:|
| Original PairwiseGRU | 71.10% |
| Shuffled observation 1 | 69.40% |
| Shuffled observation 2 | 68.50% |
| Shuffled both observations | 67.40% |
| Hypothesis-only model | 69.10% |

Across 10 permutations, prediction-change rates averaged 4.80% for O1, 11.99% for O2, and 13.11% for both observations.

The full model outperformed the hypothesis-only baseline by 2.00 percentage points. However, the 95% bootstrap interval [-0.70, 4.70] includes zero.

**Conclusion:** The model shows sensitivity to contextual observations, especially O2. However, the strong hypothesis-only performance suggests that hypothesis-level patterns contribute substantially to its predictions. The context ablations are diagnostic because shuffled observations create unnatural inputs.

### Context Sensitivity Results

| Experiment | Validation accuracy |
|---|---:|
| Original PairwiseGRU | 71.10% |
| Shuffle observation 1 | 69.40% |
| Shuffle observation 2 | 68.50% |
| Shuffle both observations | 67.40% |
| Hypothesis-only PairwiseGRU | 69.10% |

Across 10 repeated context permutations, the average prediction-change rates were:
- Observation 1: 4.80%
- Observation 2: 11.99%
- Both observations: 13.11%

The model is more sensitive to changes in observation 2 than observation 1.

### Hypothesis-Only Comparison

The hypothesis-only model reached 69.10% validation accuracy, compared with 71.10% for the full PairwiseGRU.

Paired error analysis:
- Both correct: 606 examples
- Full model only correct: 105
- Hypothesis-only model only correct: 85
- Both wrong: 204
- Prediction agreement: 81.00%

The full model's advantage was +2.00 percentage points, with a 95% bootstrap interval of [-0.70, 4.70] percentage points.

### Conclusion

The PairwiseGRU uses contextual information, as shown by changes in accuracy and predictions when observations are replaced.

However, the hypothesis-only model achieves almost the same validation accuracy. This suggests that hypothesis-level patterns provide a substantial part of the predictive signal.

The bootstrap interval includes zero, so the observed accuracy advantage of the full model is not conclusive.

Context shuffling creates unnatural inputs, so these experiments are diagnostic rather than definitive measures of causal reasoning.